<a href="https://colab.research.google.com/github/fabriciosfalchet/Projeto-Oraculo-Da-Copa/blob/main/ProjetoCopaDoMundo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# ==========================================
# 1. PREPARAÇÃO DOS DADOS
# ==========================================
# Lendo a base do Kaggle (certifique-se de que o arquivo train.csv está na pasta)
# NOTE: Please adjust the path to your 'train.csv' file if it's not directly in MyDrive.
df = pd.read_csv('/content/train.csv').dropna()

# Separando o gabarito (Target) das características (Features)
y = df['result']
X_bruto = df[['home_team', 'away_team', 'tournament', 'neutral', 'home_conf', 'away_conf']]

# Transformando nomes de times e continentes em números (One-Hot Encoding)
X = pd.get_dummies(X_bruto)

# Dividindo 70% para a IA treinar e 30% para a gente testar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## ==========================================
# 2. MODELAGEM INDIVIDUAL E O COMITÊ
# ==========================================
# Criando os 3 juízes (modelos) exigidos [cite: 31, 32, 35, 36]
knn = KNeighborsClassifier(n_neighbors=5)
arvore = DecisionTreeClassifier(random_state=42)
rede_neural = MLPClassifier(random_state=42, max_iter=500)

# Dicionário prático para passar por cada modelo
modelos_individuais = {
    'KNN': knn,
    'Árvore de Decisão': arvore,
    'Rede Neural': rede_neural
}

print("="*50)
print("📊 AVALIAÇÃO INDIVIDUAL DOS MODELOS 📊")
print("="*50)

# Treinando e testando um por um separadamente [cite: 38, 39]
for nome, modelo in modelos_individuais.items():
    modelo.fit(X_train, y_train)              # O modelo estuda a base
    previsao_ind = modelo.predict(X_test)     # O modelo faz a prova
    acuracia_ind = accuracy_score(y_test, previsao_ind)
    print(f"Acurácia {nome}: {acuracia_ind * 100:.2f}%")

print("\n" + "="*50)
print(" AVALIAÇÃO DO COMITÊ ")
print("="*50)

# Construindo o modelo ensemble combinando os classificadores [cite: 40, 41]
comite = VotingClassifier(estimators=[
    ('KNN', knn),
    ('Árvore de Decisão', arvore),
    ('Rede Neural', rede_neural)
], voting='hard')

# Treinando e testando o Comitê inteiro
comite.fit(X_train, y_train)
previsao_comite_teste = comite.predict(X_test)
acuracia_comite = accuracy_score(y_test, previsao_comite_teste)

print(f"Acurácia Comitê (Votação Majoritária): {acuracia_comite * 100:.2f}%")
print("="*50)

# ==========================================
# 3. O GRAN FINALE: PREVENDO A COPA DE 2026
# ==========================================
# Escalando os times da final (Pode trocar 'Brazil' e 'France' se quiser testar outros)
jogo_final = pd.DataFrame({
    'home_team': ['Brazil'],        # Mandante
    'away_team': ['Spain'],        # Visitante
    'tournament': ['FIFA World Cup'],
    'neutral': [True],
    'home_conf': ['CONMEBOL'],
    'away_conf': ['UEFA']
})

# Preparando os dados da final para a IA ler
jogo_final_dummies = pd.get_dummies(jogo_final)
jogo_final_pronto = jogo_final_dummies.reindex(columns=X.columns, fill_value=0)

# Perguntando ao Comitê
previsao_comite = comite.predict(jogo_final_pronto)
time_mandante = jogo_final['home_team'].values[0]
time_visitante = jogo_final['away_team'].values[0]

print("\n" + "="*50)
print("🔮 PREVISÃO DA GRANDE FINAL 🔮")
print("="*50)
print(f"JOGO: {time_mandante} x {time_visitante}\n")

print("🗣️ O QUE CADA MODELO ACHOU:")
nomes_modelos = ['KNN', 'Árvore de Decisão', 'Rede Neural']

# Descobrindo o voto individual de cada juiz
for nome, modelo in zip(nomes_modelos, comite.estimators_):
    voto = modelo.predict(jogo_final_pronto)[0]

    # Traduzindo o código da máquina para o nome da seleção
    if voto == 'H' or voto == 2:
        voto_traduzido = time_mandante
    elif voto == 'A' or voto == 0:
        voto_traduzido = time_visitante
    else:
        voto_traduzido = "Empate"

    print(f"   - {nome} votou em: {voto_traduzido}")

# Exibindo o Veredito do Comitê
print("\n" + "-"*50)
print("⚖️ VEREDITO DO COMITÊ (VOTAÇÃO MAJORITÁRIA):")

# Traduzindo o voto final do comitê
if previsao_comite[0] == 'H' or previsao_comite[0] == 2:
    print(f"VENCEDOR FINAL: 🏆 {time_mandante} (Avança para a glória na Copa!)")
elif previsao_comite[0] == 'A' or previsao_comite[0] == 0:
    print(f"VENCEDOR FINAL: 🏆 {time_visitante} (Fez história e levou a vitória!)")
else:
    print("RESULTADO FINAL: ⚖️ EMPATE! (Haja coração, a decisão vai para os pênaltis!)")
print("="*50)

📊 AVALIAÇÃO INDIVIDUAL DOS MODELOS 📊
Acurácia KNN: 42.97%
Acurácia Árvore de Decisão: 43.00%
Acurácia Rede Neural: 44.51%

 AVALIAÇÃO DO COMITÊ 
Acurácia Comitê (Votação Majoritária): 43.44%

🔮 PREVISÃO DA GRANDE FINAL 🔮
JOGO: Brazil x Spain

🗣️ O QUE CADA MODELO ACHOU:
   - KNN votou em: Spain
   - Árvore de Decisão votou em: Brazil
   - Rede Neural votou em: Spain

--------------------------------------------------
⚖️ VEREDITO DO COMITÊ (VOTAÇÃO MAJORITÁRIA):
VENCEDOR FINAL: 🏆 Spain (Fez história e levou a vitória!)
